# Homework 5 : Deployment

<div class="alert alert-block alert-info">
<b>NOTE:</b><br>
This homework uses the pinned 2026 lead-scoring release in the course repository.
The plan and report are available in <b><code>cohorts/2026/data/</code></b>.
</div>


In this homework, we will use the 2026 lead-scoring dataset. Download it from [here](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/main/cohorts/2026/data/course_lead_scoring_2026.csv).
Or you can do it with `wget`, in case don't have the ´data´:


In [1]:
#data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/main/cohorts/2026/data/course_lead_scoring_2026.csv
#!wget -N $data -P ../data/

The model was trained from
[`course_lead_scoring_2026.csv`](../../data/course_lead_scoring_2026.csv) with
the following observable features:

| Feature | Type | Missing-value rule |
| --- | --- | --- |
| `lead_source` | categorical | `NA` |
| `industry` | categorical | `NA` |
| `employment_status` | categorical | `NA` |
| `location` | categorical | `NA` |
| `number_of_courses_viewed` | integer | training-set median |
| `annual_income` | number | training-set median |
| `interaction_count` | integer | training-set median |
| `lead_score` | number | training-set median |

We use a `DictVectorizer` followed by
`LogisticRegression(solver="liblinear", C=1.0)`. The training script records
the data checksum and imputation values in `model_metadata.json`.
`lead_score` is available before conversion and `converted` is the target.

## Setup
Using Python 3.11.15 and `uv`:

In [3]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [5]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696 --lifespan off

INFO:     Started server process [73925]
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Finished server process [73925]


In [12]:
!pkill -f "uvicorn predict:app"

In another terminal:

In [7]:
!python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


## Container run

In [13]:
!docker build -t zoomcamp-model:2026-hw10 .




[+] Building 0.0s (0/1)                                           docker:colima
[+] Building 0.1s (1/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.0s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.0s
[+] Building 0.3s (1/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.2s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.2s
[+] Building 0.4s (1/3)                                           docker:colima
 => [internal] load build definition 

In [14]:
!docker run --rm -p 9696:9696 zoomcamp-model:2026-hw10

INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Finished server process [1]


The API exposes `GET /health` and `POST /predict`. The model artifact is
addressed by its checksum in `model_metadata.json`; the API health response
also reports the feature-default checksum. The Dockerfile pins the Python and
uv image digests and installs from the checked-in lockfile.

## Question 1 — environment check

Run `uv --version` and record the version in your notes. This is a setup check,
not a graded answer: your local `uv` version is not a property of the model.

In [15]:
!uv --version

uv 0.13.5 (Homebrew 2026-10-22 aarch64-apple-darwin)


## Question 2 — locked dependency

Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the
2026 reference environment?

In [17]:
!uv pip list | grep scikit-learn

scikit-learn      1.7.2


## Question 3 — load the model

Write a Python script that loads `pipeline.bin` with `pickle` and calls
`predict_proba` for this lead:

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

Report the probability of conversion rounded to three decimal places. The
grader accepts an absolute error of at most `0.005`; do not choose the nearest
answer from a list.

In [1]:
!uv run python q3_script.py

0.77


## Question 4 — serve the model

Start the provided API:

In [3]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696 --lifespan off

INFO:     Started server process [73152]
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Finished server process [73152]


The API has `GET /health` and `POST /predict`. Send this second lead:

```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

In [4]:
!curl -s http://localhost:9696/predict -H 'Content-Type: application/json' -d '{"lead_source": "organic_search", "industry": "technology", "employment_status": "employed", "location": "europe", "number_of_courses_viewed": 4, "annual_income": 80304.0, "interaction_count": 7, "lead_score": 0.74}'

{"conversion_probability":0.769799,"conversion":true}

## Question 5 — look at the container configuration

Which Python base-image tag is declared in the Dockerfile?

The version is : - `python:3.11.15-slim-bookworm`

### Container

The checked-in `Dockerfile` is the canonical 2026 container configuration. It uses
Python `3.11.15-slim-bookworm`, `uv 0.10.11`, and the checked-in `uv.lock`.
It copies the frozen artifact and starts the API on port `9696`.

Build and run it:

In [8]:
!docker build -t zoomcamp-model:2026-hw5 .


[+] Building 0.0s (0/1)                                           docker:colima
[+] Building 0.2s (1/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.2s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.2s
[+] Building 0.3s (1/3)                                           docker:colima
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 637B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11.15-slim-bo  0.3s
 => [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472  0.3s
[+] Building 0.5s (1/3)                                           docker:colima
 => [internal] load build definition fr

In [10]:
!docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5 --lifespan off

INFO:     Started server process [1]
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Finished server process [1]


## Question 6 — run the container

Run the same request from Question 4 against the container. You can use the
provided smoke client:

In [11]:
!python3 q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


Report `conversion_probability` rounded to three decimal places. The expected
result is the same reference inference as Question 4, with the same `0.005`
absolute tolerance. Also verify that:

In [12]:
!curl -s http://localhost:9696/health

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}

returns `{"status":"ok", ...}` and the model checksum recorded in
`model_metadata.json`.

The conversion_probability is: `0.769799`